# UAV-Based Crack Detection — YOLOv8 Training Pipeline

Complete pipeline to train a YOLOv8 model for crack detection using drone imagery.

## Pipeline Overview
1. Install dependencies
2. Download UAV crack dataset from Kaggle
3. Diagnose dataset structure
4. Preview image + mask pair
5. Convert segmentation masks to YOLO labels
6. Verify conversion visually
7. Build train/val/test splits
8. Create dataset.yaml
9. Train YOLOv8
10. Evaluate on test set
11. Run inference on sample images
12. Plot training curves
13. Download trained weights

**Dataset:** [UAV-Based Crack Detection](https://www.kaggle.com/datasets/ziya07/uav-based-crack-detection-dataset)  
**Model:** YOLOv8s (detection) or YOLOv8s-seg (segmentation)  
**Runtime:** GPU — go to Runtime > Change runtime type > T4 GPU before running.

---
## Cell 1 — Install Dependencies

In [ ]:
!pip install ultralytics kagglehub opencv-python-headless --quiet

import kagglehub
import os, shutil, yaml, random, cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from ultralytics import YOLO

print('All dependencies installed.')

---
## Cell 2 — Download Dataset from Kaggle

In [ ]:
path = kagglehub.dataset_download('ziya07/uav-based-crack-detection-dataset')
print('Dataset downloaded to:', path)

---
## Cell 3 — Diagnose Dataset Structure

In [ ]:
print('=== RAW DATASET STRUCTURE ===')
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    if level > 3:
        continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files[:5]:
        print(f'{indent}  {f}')

print('\n=== FILE TYPES SUMMARY ===')
all_files = list(Path(path).rglob('*.*'))
ext_count = Counter(f.suffix.lower() for f in all_files)
for ext, count in ext_count.most_common():
    print(f'  {ext}: {count} files')

---
## Cell 4 — Preview Image and Mask Pair

In [ ]:
dataset_root = Path(path) / 'UAV-based crack dataset used for segmentation'
image_dir    = dataset_root / 'image'
mask_dir     = dataset_root / 'masks'

sample_name = list(image_dir.glob('*.png'))[0].name
img  = cv2.imread(str(image_dir / sample_name))
mask = cv2.imread(str(mask_dir  / sample_name), cv2.IMREAD_GRAYSCALE)

print(f'Image shape : {img.shape}')
print(f'Mask  shape : {mask.shape}')
print(f'Mask unique values: {np.unique(mask)}')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original Image')
axes[1].imshow(mask, cmap='gray');                    axes[1].set_title('Crack Mask')
axes[2].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[2].imshow(mask, cmap='Reds', alpha=0.4);         axes[2].set_title('Overlay')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---
## Cell 5 — Convert Masks to YOLO Labels

The dataset provides binary mask PNGs (white = crack, black = background).  
We convert these into:
- **Detection** — bounding box `.txt` files (`class cx cy w h`)
- **Segmentation** — polygon `.txt` files (`class x1 y1 x2 y2 ...`)

Set `TASK = 'detect'` or `TASK = 'segment'` in Cell 7.

In [ ]:
dataset_root = Path(path) / 'UAV-based crack dataset used for segmentation'
image_dir    = dataset_root / 'image'
mask_dir     = dataset_root / 'masks'

OUT_DETECT_LABELS = Path('/content/labels_detect')
OUT_SEG_LABELS    = Path('/content/labels_segment')
OUT_DETECT_LABELS.mkdir(exist_ok=True)
OUT_SEG_LABELS.mkdir(exist_ok=True)

converted = empty = skipped = 0

for mask_path in sorted(mask_dir.glob('*.png')):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        skipped += 1
        continue

    H, W = mask.shape
    if mask.max() <= 1:
        mask = (mask * 255).astype(np.uint8)

    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    stem = mask_path.stem

    if len(contours) == 0:
        open(OUT_DETECT_LABELS / f'{stem}.txt', 'w').close()
        open(OUT_SEG_LABELS    / f'{stem}.txt', 'w').close()
        empty += 1
        continue

    detect_lines, seg_lines = [], []

    for contour in contours:
        if cv2.contourArea(contour) < 10:
            continue

        # Detection: bounding box
        x, y, w, h = cv2.boundingRect(contour)
        cx, cy = (x + w/2)/W, (y + h/2)/H
        nw, nh  = w/W, h/H
        detect_lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')

        # Segmentation: simplified polygon
        epsilon = 0.002 * cv2.arcLength(contour, True)
        approx  = cv2.approxPolyDP(contour, epsilon, True)
        pts     = approx.reshape(-1, 2)
        if len(pts) < 3:
            continue
        coords = ' '.join(f'{p[0]/W:.6f} {p[1]/H:.6f}' for p in pts)
        seg_lines.append(f'0 {coords}')

    with open(OUT_DETECT_LABELS / f'{stem}.txt', 'w') as f:
        f.write('\n'.join(detect_lines))
    with open(OUT_SEG_LABELS / f'{stem}.txt', 'w') as f:
        f.write('\n'.join(seg_lines))
    converted += 1

print(f'Converted        : {converted}')
print(f'Empty (no crack) : {empty}')
print(f'Skipped          : {skipped}')

---
## Cell 6 — Verify Conversion Visually

In [ ]:
labeled = [f for f in OUT_DETECT_LABELS.glob('*.txt') if f.stat().st_size > 0]
sample  = random.choice(labeled)

img_path = image_dir / (sample.stem + '.png')
img      = cv2.imread(str(img_path))
H, W     = img.shape[:2]

n_boxes = 0
for line in open(sample).read().strip().split('\n'):
    parts = line.split()
    if len(parts) < 5:
        continue
    _, cx, cy, nw, nh = map(float, parts)
    x1 = int((cx - nw/2) * W);  y1 = int((cy - nh/2) * H)
    x2 = int((cx + nw/2) * W);  y2 = int((cy + nh/2) * H)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
    n_boxes += 1

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f'Verification: {sample.stem}  -  {n_boxes} crack(s)')
plt.axis('off'); plt.show()

---
## Cell 7 — Build Train / Val / Test Splits

Split ratio: **80% train / 10% val / 10% test**  
Only images with at least one crack label are included.

In [ ]:
# Choose task: 'detect' for bounding boxes, 'segment' for polygon masks
TASK = 'detect'

LABELS_SRC = OUT_DETECT_LABELS if TASK == 'detect' else OUT_SEG_LABELS
BASE       = Path('/content/crack_dataset')

for split in ['train', 'val', 'test']:
    for sub in ['images', 'labels']:
        shutil.rmtree(BASE / split / sub, ignore_errors=True)
        (BASE / split / sub).mkdir(parents=True)

pairs = []
for lbl in LABELS_SRC.glob('*.txt'):
    if lbl.stat().st_size == 0:
        continue
    img = image_dir / (lbl.stem + '.png')
    if img.exists():
        pairs.append((img, lbl))

print(f'Valid image+label pairs: {len(pairs)}')

random.seed(42)
random.shuffle(pairs)
n = len(pairs)
splits = {
    'train': pairs[:int(n * 0.8)],
    'val'  : pairs[int(n * 0.8):int(n * 0.9)],
    'test' : pairs[int(n * 0.9):]
}

for split, split_pairs in splits.items():
    for img_path, lbl_path in split_pairs:
        shutil.copy(img_path, BASE / split / 'images' / img_path.name)
        shutil.copy(lbl_path, BASE / split / 'labels' / lbl_path.name)

print('\nSplit verification:')
for split in ['train', 'val', 'test']:
    n_img = len(list((BASE / split / 'images').glob('*')))
    n_lbl = len(list((BASE / split / 'labels').glob('*.txt')))
    ok    = 'OK' if n_img == n_lbl else 'MISMATCH'
    print(f'  {split}: {n_img} images, {n_lbl} labels  [{ok}]')

---
## Cell 8 — Create dataset.yaml

In [ ]:
dataset_yaml = {
    'path' : str(BASE),
    'train': 'train/images',
    'val'  : 'val/images',
    'test' : 'test/images',
    'nc'   : 1,
    'names': ['crack']
}
yaml_path = BASE / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print('dataset.yaml:')
print(open(yaml_path).read())

---
## Cell 9 — Train YOLOv8

| Model | Speed | Accuracy |
|-------|-------|----------|
| yolov8n.pt | Fastest | Lower |
| yolov8s.pt | Fast | Good |
| yolov8m.pt | Slower | Better |

In [ ]:
model_weights = 'yolov8s-seg.pt' if TASK == 'segment' else 'yolov8s.pt'
model = YOLO(model_weights)

results = model.train(
    data     = str(yaml_path),
    epochs   = 50,      # increase to 100 for better accuracy
    imgsz    = 640,
    batch    = 16,      # reduce to 8 if you get CUDA out-of-memory
    name     = f'crack_{TASK}_v1',
    project  = '/content/runs',
    device   = 0,
    patience = 10,
    save     = True,
    plots    = True,
)

print('Training complete!')
print('Weights saved at:', results.save_dir)

---
## Cell 10 — Evaluate on Test Set

In [ ]:
metrics = model.val(data=str(yaml_path), split='test')

print('\n=== Test Set Metrics ===')
print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'mAP50-95 : {metrics.box.map:.4f}')

p_vals = metrics.box.p
r_vals = metrics.box.r
if hasattr(p_vals, '__len__') and len(p_vals) > 0:
    print(f'Precision: {np.nanmean(p_vals):.4f}')
    print(f'Recall   : {np.nanmean(r_vals):.4f}')

if TASK == 'segment' and hasattr(metrics, 'seg'):
    print(f'Seg mAP50: {metrics.seg.map50:.4f}')

---
## Cell 11 — Run Inference on Sample Test Images

In [ ]:
test_imgs = list((BASE / 'test' / 'images').glob('*.png'))[:4]

pred_results = model.predict(
    source  = test_imgs,
    conf    = 0.25,
    save    = True,
    project = '/content/runs',
    name    = 'sample_predictions'
)

pred_dir  = Path('/content/runs/sample_predictions')
pred_imgs = list(pred_dir.glob('*.png'))[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, p in enumerate(pred_imgs):
    img = cv2.imread(str(p))
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'Prediction {i+1}')
    axes[i].axis('off')
plt.tight_layout(); plt.show()

---
## Cell 12 — Plot Training Curves

In [ ]:
import pandas as pd

results_csv = Path(f'/content/runs/crack_{TASK}_v1/results.csv')

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    if 'train/box_loss' in df.columns:
        axes[0].plot(df['epoch'], df['train/box_loss'], label='Train')
        axes[0].plot(df['epoch'], df['val/box_loss'],   label='Val')
        axes[0].set_title('Box Loss'); axes[0].legend()

    if 'metrics/mAP50(B)' in df.columns:
        axes[1].plot(df['epoch'], df['metrics/mAP50(B)'], color='green')
        axes[1].set_title('mAP50')

    if 'metrics/precision(B)' in df.columns:
        axes[2].plot(df['epoch'], df['metrics/precision(B)'], label='Precision')
        axes[2].plot(df['epoch'], df['metrics/recall(B)'],    label='Recall')
        axes[2].set_title('Precision & Recall'); axes[2].legend()

    for ax in axes: ax.set_xlabel('Epoch')
    plt.tight_layout(); plt.show()
else:
    print('results.csv not found.')

---
## Cell 13 — Download Trained Weights

Downloads `best.pt` to your local machine.  
Place it in the same folder as `app.py` to run the Gradio interface.

In [ ]:
from google.colab import files

best_pt = f'/content/runs/crack_{TASK}_v1/weights/best.pt'
last_pt = f'/content/runs/crack_{TASK}_v1/weights/last.pt'

print(f'best.pt size: {os.path.getsize(best_pt)/1e6:.1f} MB')

files.download(best_pt)   # use this in app.py
files.download(last_pt)   # backup

---
## Cell 14 — (Optional) Save to Google Drive

In [ ]:
# Uncomment to mount Drive and save weights there

# from google.colab import drive
# drive.mount('/content/drive')
#
# save_dir = Path('/content/drive/MyDrive/crack_detection')
# save_dir.mkdir(parents=True, exist_ok=True)
#
# shutil.copy(best_pt, save_dir / 'best.pt')
# shutil.copy(last_pt, save_dir / 'last.pt')
# shutil.copy(str(yaml_path), save_dir / 'dataset.yaml')
# print('Saved to Google Drive:', save_dir)

---
## Summary

| Step | Output |
|------|--------|
| Dataset download | Kaggle UAV crack images + masks |
| Mask conversion | YOLO `.txt` label files |
| Train/val/test split | 80 / 10 / 10 |
| Trained model | `best.pt` |
| Evaluation | mAP50, mAP50-95, Precision, Recall |

### Next step
Place `best.pt` in your VS Code project alongside `app.py` and run:
```bash
python app.py
```
Then open `http://127.0.0.1:7860`